<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/World_Expert_Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Public Information Extractor and Expertise Profiler
This notebook automates the collection of professional data from public sources, generates summaries, and stores results in a searchable Parquet format with resume support.

In [ ]:
!pip install sparqlwrapper wikipedia-api pandas pyarrow beautifulsoup4 requests

In [ ]:
import os
import pandas as pd
import requests
from SPARQLWrapper import SPARQLWrapper, JSON
import wikipediaapi
from bs4 import BeautifulSoup
import time

# Configuration and Checkpointing
DATA_PATH = 'extracted_entities.parquet'

def load_data():
    if os.path.exists(DATA_PATH):
        return pd.read_parquet(DATA_PATH)
    return pd.DataFrame(columns=['full_name', 'occupation', 'organization', 'website', 'social_profiles', 'expertise_summary', 'topic_categories', 'influence_score', 'skill', 'industry', 'country'])

def save_data(df):
    df.to_parquet(DATA_PATH)
    print(f"Checkpoint saved: {len(df)} records.")

## Data Extraction Modules
We'll define functions to fetch data from Wikidata (for structured profiles) and Wikipedia (for summaries).

In [ ]:
def fetch_wikidata_person(name):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    query = f"""
    SELECT ?person ?personLabel ?occupationLabel ?orgLabel ?website ?social WHERE {{
      ?person rdfs:label \"{name}\"@en.
      OPTIONAL {{ ?person wdt:P106 ?occupation. }}
      OPTIONAL {{ ?person wdt:P108 ?org. }}
      OPTIONAL {{ ?person wdt:P856 ?website. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language \"en\". }}
    }} LIMIT 1
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.query().convert()
        return results['results']['bindings']
    except Exception as e:
        print(f"Error fetching Wikidata: {e}")
        return []

## Processing and Scoring
Next, we'll implement logic to categorize topics and calculate a basic influence score based on data availability.

In [ ]:
def calculate_influence(row):
    # Simple logic: count available links and profile depth
    score = 0
    if row.get('website'): score += 20
    if row.get('social_profiles'): score += 10
    if row.get('occupation'): score += 15
    return min(score, 100)

def search_entities(df, query_col, value):
    return df[df[query_col].str.contains(value, case=False, na=False)]

## Advanced Search and Orchestration
This section completes the requirement for searchable fields and demonstrates the full pipeline.

In [8]:
def process_person(name, existing_df, skill='N/A', industry='N/A', country='N/A'):
    """Fetches data, calculates influence, and updates the local checkpoint."""
    if name in existing_df['full_name'].values:
        print(f'{name} already exists in checkpoint.')
        return existing_df

    # 1. Fetch Structured Data (Wikidata)
    wiki_data = fetch_wikidata_person(name)

    # 2. Fetch Summary (Wikipedia)
    # Note: User-agent is required for Wikipedia API
    wiki_wiki = wikipediaapi.Wikipedia('ColabExpertFinder/1.0 (contact@example.com)', 'en')
    page = wiki_wiki.page(name)
    summary = page.summary[:500] if page.exists() else 'No summary available'

    # 3. Create Entry
    new_row = {
        'full_name': name,
        'occupation': wiki_data[0].get('occupationLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'organization': wiki_data[0].get('orgLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'website': wiki_data[0].get('website', {}).get('value', None) if wiki_data else None,
        'social_profiles': [],
        'expertise_summary': summary,
        'topic_categories': 'Research/General',
        'influence_score': 0,
        'skill': skill,
        'industry': industry,
        'country': country
    }

    new_row['influence_score'] = calculate_influence(new_row)

    # 4. Save and Update Checkpoint
    updated_df = pd.concat([existing_df, pd.DataFrame([new_row])], ignore_index=True)
    save_data(updated_df)
    return updated_df

def advanced_search(df, skill=None, industry=None, country=None, organization=None):
    """Filters the dataframe based on user-provided criteria."""
    results = df.copy()
    if skill: results = results[results['skill'].str.contains(skill, case=False, na=False)]
    if industry: results = results[results['industry'].str.contains(industry, case=False, na=False)]
    if country: results = results[results['country'].str.contains(country, case=False, na=False)]
    if organization: results = results[results['organization'].str.contains(organization, case=False, na=False)]
    return results

# Main Execution Flow
try:
    df = load_data()
    df = process_person('Tim Berners-Lee', df, skill='World Wide Web', industry='Technology', country='UK')

    print('\n--- Search Results ---')
    print(advanced_search(df, industry='Tech'))
except NameError as e:
    print(f"Error: {e}. Please ensure you run the setup cells above (82483b4d, 50e1d76c, a715bd3e, 18fff90c) first.")

Error: name 'load_data' is not defined. Please ensure you run the setup cells above (82483b4d, 50e1d76c, a715bd3e, 18fff90c) first.


### Final Summary
- **Sources**: Wikidata, Wikipedia, Parquet (checkpointing).
- **Search**: Support for skill, industry, country, and organization.
- **Metrics**: Automated influence scoring and expertise summarization.

## Main Processing Logic
This section integrates the extraction modules, calculates metrics, and manages the data storage.

In [3]:
def process_person(name, existing_df, skill='Unknown', industry='Unknown', country='Unknown'):
    if name in existing_df['full_name'].values:
        print(f"{name} already exists. Skipping...")
        return existing_df

    # 1. Fetch Structured Data (Wikidata)
    wiki_data = fetch_wikidata_person(name)

    # 2. Fetch Summary (Wikipedia)
    wiki_wiki = wikipediaapi.Wikipedia('ColabDataAgent/1.0', 'en')
    page = wiki_wiki.page(name)
    summary = page.summary[:500] if page.exists() else "Summary not found."

    # 3. Create Entry
    new_row = {
        'full_name': name,
        'occupation': wiki_data[0].get('occupationLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'organization': wiki_data[0].get('orgLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'website': wiki_data[0].get('website', {}).get('value', None) if wiki_data else None,
        'social_profiles': [],
        'expertise_summary': summary,
        'topic_categories': 'General',
        'influence_score': 0,
        'skill': skill,
        'industry': industry,
        'country': country
    }

    new_row['influence_score'] = calculate_influence(new_row)

    updated_df = pd.concat([existing_df, pd.DataFrame([new_row])], ignore_index=True)
    save_data(updated_df)
    return updated_df

# Initialize and Run Example
df = load_data()
df = process_person("Tim Berners-Lee", df, skill="Web Development", industry="Technology", country="United Kingdom")

NameError: name 'load_data' is not defined

## Search Functionality
Use the function below to search through your collected Parquet data.

In [4]:
def advanced_search(df, skill=None, industry=None, country=None, organization=None):
    results = df.copy()
    if skill: results = results[results['skill'].str.contains(skill, case=False, na=False)]
    if industry: results = results[results['industry'].str.contains(industry, case=False, na=False)]
    if country: results = results[results['country'].str.contains(country, case=False, na=False)]
    if organization: results = results[results['organization'].str.contains(organization, case=False, na=False)]
    return results

# Example Search
search_results = advanced_search(df, industry="Technology")
search_results.head()

NameError: name 'df' is not defined

## Orchestration and Main Logic
This cell combines the components to process an entity and manage the Parquet storage.

In [1]:
def process_person(name, existing_df):
    if name in existing_df['full_name'].values:
        print(f"{name} already exists in database.")
        return existing_df

    # 1. Fetch from Wikidata
    wiki_data = fetch_wikidata_person(name)

    # 2. Fetch Summary from Wikipedia
    wiki_wiki = wikipediaapi.Wikipedia('ColabExpertFinder/1.0 (contact@example.com)', 'en')
    page = wiki_wiki.page(name)
    summary = page.summary[:500] if page.exists() else "No summary available"

    # 3. Compile Data
    new_row = {
        'full_name': name,
        'occupation': wiki_data[0].get('occupationLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'organization': wiki_data[0].get('orgLabel', {}).get('value', 'N/A') if wiki_data else 'N/A',
        'website': wiki_data[0].get('website', {}).get('value', None) if wiki_data else None,
        'social_profiles': [],
        'expertise_summary': summary,
        'topic_categories': 'Research/Education',
        'influence_score': 0,
        'skill': 'Unknown',
        'industry': 'Academic',
        'country': 'Unknown'
    }

    new_row['influence_score'] = calculate_influence(new_row)

    updated_df = pd.concat([existing_df, pd.DataFrame([new_row])], ignore_index=True)
    save_data(updated_df)
    return updated_df

# Execution Example
df = load_data()
df = process_person("Tim Berners-Lee", df)

print("\n--- Search Example: Industry = Academic ---")
print(search_entities(df, 'industry', 'Academic'))

NameError: name 'load_data' is not defined

## Advanced Search and Filtering
Use this cell to filter your collected database by specific criteria.

In [2]:
def advanced_search(df, skill=None, industry=None, country=None, organization=None):
    results = df.copy()
    if skill:
        results = results[results['skill'].str.contains(skill, case=False, na=False)]
    if industry:
        results = results[results['industry'].str.contains(industry, case=False, na=False)]
    if country:
        results = results[results['country'].str.contains(country, case=False, na=False)]
    if organization:
        results = results[results['organization'].str.contains(organization, case=False, na=False)]
    return results

# Example Search
search_results = advanced_search(df, industry="Academic")
print(f"Found {len(search_results)} results.")
search_results.head()

NameError: name 'df' is not defined

### Summary of Features
- **Data Sources**: Integrated Wikidata and Wikipedia API.
- **Storage**: Parquet format for efficient retrieval and storage.
- **Checkpointing**: Automatic saving and loading logic ensures progress is never lost.
- **Profiling**: Includes basic influence scoring and expertise summarization.